In [ ]:
import numpy as np 
import pandas as pd 
import MLM.angle_between as angle
import MLM.moire_lattice_vector_finder as mlf 
import MLM.structure_writer as sw
from matplotlib.path import Path
from MLM.match import run_and_filter
from ase import Atoms
from ase.io import write

In [2]:
file_1 = "../moire_structures/mos2/mos2_monolayer.vasp"
file_2 = "../moire_structures/mos2/mos2_monolayer.vasp"
file_3 = "../moire_structures/mos2/mos2_monolayer.vasp"

In [3]:
lattice_vectors1, atom_type_arr1, dat1 = mlf.read_vasp(file_1)

lattice_vectors2, atom_type_arr2, dat2 = mlf.read_vasp(file_2)

lattice_vectors3, atom_type_arr3, dat3 = mlf.read_vasp(file_3)


In [4]:
candidate = pd.read_pickle('../moire_structures/mos2/mos2_ttl.pkl')

candidate

,Theta,Phi,A1,A2,delvec,delang,norm_A1,norm_A2
10,21.79,38.21,"[11.080997057200001, 19.192852852599998]","[-11.081002168600001, 19.192852852599998]",0.001244,3.814877e-06,22.161997,22.161999
11,21.79,43.57,"[3.1659969686, 21.9346889744]","[20.5789981858, 8.2255083654]",0.001387,7.318076e-06,22.161997,22.161998
17,21.79,49.58,"[17.4129961058, 24.6765250962]","[-12.6640032086, 27.418361217999998]",0.001352,4.946715e-06,30.201711,30.201714
18,27.80,49.58,"[17.4129961058, 24.6765250962]","[30.076999314400005, -2.7418361218000005]",0.001352,2.557022e-06,30.201711,30.201714
19,21.79,53.99,"[1.5829959286, 30.1601973398]","[26.9109972344, 13.709180609]",0.000540,7.587588e-06,30.201712,30.201712
20,32.20,53.99,"[-1.5829959286000008, -30.1601973398]","[25.3280013058, -16.4510167308]",0.000540,3.101992e-06,30.201712,30.201715
37,21.79,34.96,"[7.9149949772, 35.6438695834]","[34.825997323, 10.967344487200002]",0.000220,6.912406e-06,36.512088,36.512090
38,13.17,34.96,"[7.9149949772, 35.643869583400004]","[-26.9110023458, 24.676525096200002]",0.000220,6.595630e-07,36.512088,36.512093
43,27.80,55.59,"[1.582994468199999, 41.127541827]","[36.4089961724, 19.192852852599998]",0.001115,7.606940e-06,41.157995,41.157996
53,38.21,56.11,"[1.5829937379999999, 46.6112140706]","[41.1579956414, 21.9346889744]",0.000188,7.611934e-06,46.638087,46.638087


In [5]:
df = candidate

In [ ]:
import warnings
warnings.filterwarnings('ignore')


count = 1
for row in df.iterrows():
    A1 = np.array(row[1]['A1'])
    A2 = np.array(row[1]['A2'])
    delvec = row[1]['delvec']
    print("Delvec is ", delvec)
    norm = row[1]['norm_A1'] + row[1]['norm_A2']
    replicate = int((norm/(min(np.linalg.norm(lattice_vectors1['a']),np.linalg.norm(lattice_vectors2['a'])))))*8

    new_lattice_vectors1, new_total_atoms1, replicated_atom1, replicated_type_arr1 = sw.replicate_atoms(a = lattice_vectors1['a'],
                                                                                             b = lattice_vectors1['b'],
                                                                                             c = lattice_vectors1['c'],
                                                                                             atom_data = dat1,
                                                                                             atom_type_arr = atom_type_arr1,
                                                                                             natoms = len(dat1),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)
    new_lattice_vectors2, new_total_atoms2, replicated_atom2, replicated_type_arr2 = sw.replicate_atoms(a = lattice_vectors2['a'],
                                                                                             b = lattice_vectors2['b'],
                                                                                             c = lattice_vectors2['c'],
                                                                                             atom_data = dat2,
                                                                                             atom_type_arr = atom_type_arr2,
                                                                                             natoms = len(dat2),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)
    new_lattice_vectors3, new_total_atoms3, replicated_atom3, replicated_type_arr3 = sw.replicate_atoms(a = lattice_vectors3['a'],
                                                                                             b = lattice_vectors3['b'],
                                                                                             c = lattice_vectors3['c'],
                                                                                             atom_data = dat3,
                                                                                             atom_type_arr = atom_type_arr3,
                                                                                             natoms = len(dat3),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)

    # Create a DataFrame for positions
    positions1_df = pd.DataFrame(replicated_atom1, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions1_df['type'] = replicated_type_arr1
    
    positions2_df = pd.DataFrame(replicated_atom2, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions2_df['type'] = replicated_type_arr2 
    positions2_df['z'] = positions2_df['z'] + 3 + (positions1_df['z'].max() )
    
    positions3_df = pd.DataFrame(replicated_atom3, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions3_df['type'] = replicated_type_arr3 
    positions3_df['z'] = positions3_df['z'] + 3 + (positions2_df['z'].max() )
    

    # Rotating the second or middle layer
    theta1 = row[1]['Theta']
    
    theta1_rad = np.deg2rad(theta1)
    
    rotation_matrix_1 = np.array([[np.cos(theta1_rad), -np.sin(theta1_rad), 0],
                            [np.sin(theta1_rad), np.cos(theta1_rad), 0],
                            [0, 0, 1]])
    
    middle_layer_positions = positions2_df[['x', 'y', 'z']].values  # Extract x, y, z positions
    middle_rotated_positions = middle_layer_positions @ rotation_matrix_1.T  # Apply rotation

    # Create a new DataFrame for the rotated positions
    rotated_middle_layer = pd.DataFrame(middle_rotated_positions, columns=['x', 'y', 'z'])

    # Add the atom types back to the rotated DataFrame
    rotated_middle_layer['type'] = positions2_df['type'].values
    
    # Rotating the top or third layer
    theta2 = row[1]['Phi']
    
    theta2_rad = np.deg2rad(theta2)
    
    rotation_matrix_2 = np.array([[np.cos(theta2_rad), -np.sin(theta2_rad), 0],
                            [np.sin(theta2_rad), np.cos(theta2_rad), 0],
                            [0, 0, 1]])
    
    top_layer_positions = positions3_df[['x', 'y', 'z']].values  # Extract x, y, z positions
    top_rotated_positions = top_layer_positions @ rotation_matrix_2.T  # Apply rotation

    # Create a new DataFrame for the rotated positions
    rotated_top_layer = pd.DataFrame(top_rotated_positions, columns=['x', 'y', 'z'])

    # Add the atom types back to the rotated DataFrame
    rotated_top_layer['type'] = positions3_df['type'].values
    
    
    
    # Concatenate DataFrames vertically
    concat_vertical = pd.concat([positions1_df, rotated_middle_layer, rotated_top_layer], ignore_index=True)
    
    z_dim = (positions3_df['z'].max() - positions1_df['z'].min()) + 50
    

    A3 = np.array([0, 0, z_dim])
    
    vertex = [ [0.0,0.0], [A1[0], A1[1]],  [(A1+A2)[0],(A1+A2)[1]], [A2[0],A2[1]]]
    
    
    
    selected_atoms_df = sw.select_atoms_fractional(concat_vertical, A1, A2, dtol=delvec)
    
    
    sw.write_lammps_ase(f"../moire_structures/mos2/TTL_mos2_{theta1}_{theta2}.dat",A1,A2,A3, selected_atoms_df)
    print(f"Done with {count} | {np.round(theta1,2)} | {np.round(theta2,2)}")
    count = count + 1
    
    
    
    


Delvec is  0.001243555749335279
Done with 1 | 21.79 | 38.21
Delvec is  0.001386549318922247
Done with 2 | 21.79 | 43.57
Delvec is  0.0013521424329268647
Done with 3 | 21.79 | 49.58
Delvec is  0.0013521424334265984
Done with 4 | 27.8 | 49.58
Delvec is  0.0005397325890714131
Done with 5 | 21.79 | 53.99
Delvec is  0.000539732593826531
Done with 6 | 32.2 | 53.99
Delvec is  0.00022014244856911365
Done with 7 | 21.79 | 34.96
Delvec is  0.00022014245006088233
Done with 8 | 13.17 | 34.96
Delvec is  0.0011150546721471846
Done with 9 | 27.8 | 55.59
Delvec is  0.00018791398304131198
Done with 10 | 38.21 | 56.11
Delvec is  0.00018791399051837758
Done with 11 | 17.9 | 56.11
Delvec is  0.0005849138454646814
Done with 12 | 27.8 | 40.97
Delvec is  0.0005849138433149471
Done with 13 | 13.17 | 40.97
